# 05 — AMX Tech Cloud-Cost Anomaly Detection
Phase 8 compares a transparent robust-IQR baseline with Isolation Forest on daily company-wide GPU cost. Set `RERUN_DETECTION = True` to recompute from validated CSV data; the default loads persisted scores for fast review.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RERUN_DETECTION = False

if RERUN_DETECTION:
    from sentinel.anomaly import detect_cloud_cost_anomalies
    from sentinel.data.loader import read_csv_dataset
    from sentinel.eda.cleaning import clean_dataset

    tables, _ = clean_dataset(read_csv_dataset(PROJECT_ROOT / 'data/generated'))
    scores, _, report = detect_cloud_cost_anomalies(tables['cloud_costs'])
else:
    report = json.loads((PROJECT_ROOT / 'models/cloud_cost_anomaly_report.json').read_text(encoding='utf-8'))
    scores = pd.read_csv(PROJECT_ROOT / 'models/cloud_cost_anomaly_scores.csv', parse_dates=['date'])

report['company'], report['configuration'], report['counts']

## Method comparison
Scenario dates are excluded from detector fitting and decision rules. They enter only in the persisted evaluation block after every score and flag has been fixed.

In [ ]:
evaluation = report['evaluation_only_ground_truth']
pd.DataFrame({
    method: {key: values[key] for key in ('precision', 'recall', 'f1')}
    for method, values in evaluation.items()
}).T.round(3)

In [ ]:
strong = scores[scores['strong_anomaly']].copy()
strong[['date', 'metric', 'normal_range_lower', 'normal_range_upper', 'observed_value', 'anomaly_score', 'interpretation']].round(3)

In [ ]:
figure = go.Figure()
figure.add_trace(go.Scatter(x=scores.date, y=scores.normal_range_upper, line={'width': 0}, showlegend=False))
figure.add_trace(go.Scatter(x=scores.date, y=scores.normal_range_lower, fill='tonexty', name='Robust normal range', line={'width': 0}))
figure.add_trace(go.Scatter(x=scores.date, y=scores.observed_value, name='Daily GPU cost', line={'color': '#2563eb'}))
figure.add_trace(go.Scatter(x=strong.date, y=strong.observed_value, mode='markers', name='Strong anomaly', marker={'color': '#dc2626', 'size': 9}))
figure.update_layout(title='AMX Tech daily GPU cost anomalies', xaxis_title='Date', yaxis_title='GPU cost', template='plotly_white')
figure

In [ ]:
scores.nlargest(12, 'anomaly_score')[['date', 'observed_value', 'relative_to_baseline', 'anomaly_score', 'status']].round(3)

## Interpretation boundary
The analysis identifies an unusual mid-May GPU-cost event, not its cause. The centered rolling median is deliberately retrospective; a production alert would require a trailing-only baseline, latency policy, drift monitoring, workload context, and business-calibrated alert costs.